# 01 Gas properties: H2 vs CH4 at Rough reservoir temperature
Purpose: quantify the volumetric energy penalty of hydrogen and validate CoolProp against a known value.

In [ ]:
# Paste as the first cell of every notebook
import subprocess, hashlib, datetime, pathlib
root = pathlib.Path.cwd().parent
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=root).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, cwd=root).stdout.strip() != ""
inp = hashlib.sha256((root / "data" / "inputs.csv").read_bytes()).hexdigest()[:10]
print(f"Run {datetime.datetime.now():%Y-%m-%d %H:%M} | commit {commit}{' (UNCOMMITTED CHANGES)' if dirty else ''} | inputs.csv {inp}")


In [ ]:
from h2_bottleneck import props
rho_h2 = props.rho(100e5, 300.0, 'Hydrogen')
print(f'H2 density at 300 K, 100 bar: {rho_h2:.2f} kg/m3 (expected ~7.8)')

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from h2_bottleneck import inputs
T = inputs.load()['reservoir_temperature']
x = inputs.load()
HHV = {"Hydrogen": x["hhv_hydrogen"], "Methane": x["hhv_methane"]}   # J/kg, Scafidi et al. (2023) Table 2
P = np.linspace(20e5, 300e5, 57)
rows = []
for gas in HHV:
    for p in P:
        r = props.rho(p, T, gas)
        rows.append(dict(gas=gas, p_bar=p/1e5, rho=r, Z=props.Z(p, T, gas),
                         mu_uPas=props.mu(p, T, gas)*1e6, GJ_per_m3=r*HHV[gas]/1e9))
df = pd.DataFrame(rows)
ratio = df[df.gas=='Hydrogen'].GJ_per_m3.values / df[df.gas=='Methane'].GJ_per_m3.values
print(f'H2/CH4 energy per m3: {ratio.min():.2f} to {ratio.max():.2f}')

In [ ]:
# Exact values at selected pressures (Rough reservoir temperature)
pressures_bar = [20, 50, 100, 150, 200, 250, 300]
rows = []
for p_bar in pressures_bar:
    p = p_bar * 1e5
    r_h2, r_ch4 = props.rho(p, T, "Hydrogen"), props.rho(p, T, "Methane")
    e_h2, e_ch4 = r_h2 * HHV["Hydrogen"] / 1e9, r_ch4 * HHV["Methane"] / 1e9   # GJ/m3
    rows.append({
        "p (bar)": p_bar,
        "H2 energy (GJ/m3)": round(e_h2, 3),
        "CH4 energy (GJ/m3)": round(e_ch4, 3),
        "H2/CH4 energy ratio": round(e_h2 / e_ch4, 3),
        "Volume multiplier": round(e_ch4 / e_h2, 2),
        "Z H2": round(props.Z(p, T, "Hydrogen"), 3),
        "Z CH4": round(props.Z(p, T, "Methane"), 3),
        "mu H2 (uPa.s)": round(props.mu(p, T, "Hydrogen") * 1e6, 2),
        "mu CH4 (uPa.s)": round(props.mu(p, T, "Methane") * 1e6, 2),
    })
summary = pd.DataFrame(rows)
summary.to_csv("../results/tables/01_gas_properties_summary.csv", index=False)
summary

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
for gas, g in df.groupby('gas'):
    ax[0].plot(g.p_bar, g.GJ_per_m3, label=gas)
    ax[1].plot(g.p_bar, g.Z, label=gas)
    ax[2].plot(g.p_bar, g.mu_uPas, label=gas)
for a, y in zip(ax, ['Energy density (GJ/m3)', 'Compressibility Z', 'Viscosity (uPa.s)']):
    a.set_xlabel('Pressure (bar)'); a.set_ylabel(y); a.legend()
fig.tight_layout(); fig.savefig('../results/figures/01_gas_properties.png', dpi=300)

## Findings
At the Rough reservoir temperature of 91.5 °C, hydrogen stores 27.4–31.4% of the energy of methane per unit volume over 20–300 bar (HHV basis), so 3.19–3.65 times the volume is needed to store the same energy. This is consistent with the approximately threefold figure reported by DESNZ (2023), but larger at reservoir pressures.

The penalty varies with pressure because the gases deviate from ideal behaviour in opposite directions. Hydrogen’s compressibility factor rises from 1.010 to 1.158, while methane’s falls to a minimum of 0.936 at 150 bar before recovering to 0.995 at 300 bar. The volume multiplier therefore peaks at 3.65 at 250 bar, close to Rough’s maximum operating pressure, so ideal-gas ratios would understate the penalty.

Hydrogen’s viscosity is low and nearly constant (10.2–10.7 µPa·s, against 13.5–21.6 µPa·s for methane), which reduces flow resistance in the wells and pipeline. Compression has no such offset. These properties make the hypothesis that compression limits injection physically plausible. The integrated model is needed to test it.